In [44]:
import pandas as pd
import re

In [52]:
corpus = pd.read_csv(r"C:\Users\utsav\Downloads\Data.csv")

## Q.5 How many records contain a decimal number?

In [53]:
import pandas as pd  
import re             

# Regular expression pattern to match decimal numbers
decimal_pattern = r'\d+\.\d+'  

# Regular expression pattern to match US-style phone numbers
phone_pattern = r"""
    (?:(?:\+?1[\s.-]?)?         # Optional country code: +1, 1, etc.
    (?:\(?\d{3}\)?[\s.-]?)      # Area code: (123), 123, etc.
    \d{3}[\s.-]?\d{4})          # Final 7 digits of the phone number
"""

# Compile the phone pattern with re.VERBOSE to allow comments and formatting
compiled_phone_pattern = re.compile(phone_pattern, re.VERBOSE)


# Function to extract all decimal numbers from text
def extract_decimals(text):
    return re.findall(decimal_pattern, text)

# Function to extract all phone numbers from text
def extract_phones(text):
    return compiled_phone_pattern.findall(text)

# Function to remove decimal numbers that appear inside phone numbers
def filter_decimals(text, decimals, phones):
    cleaned = []
    for dec in decimals:
        # Keep only those decimals that are NOT part of any phone number
        if not any(dec in phone for phone in phones):
            cleaned.append(dec)
    return cleaned

# Extract phone numbers into a new column
corpus['phone_numbers'] = corpus['Data'].apply(extract_phones)

# Extract all raw decimal numbers into a new column
corpus['raw_decimals'] = corpus['Data'].apply(extract_decimals)

# Remove decimals that are part of phone numbers and store cleaned list
corpus['valid_decimals'] = corpus.apply(
    lambda row: filter_decimals(row['Data'], row['raw_decimals'], row['phone_numbers']),
    axis=1
)

# Count how many valid decimal numbers exist in each row
corpus['decimal_count'] = corpus['valid_decimals'].apply(len)

# Filter the rows where at least one valid decimal was found
corpus_filtered = corpus[corpus['decimal_count'] > 0][['Data', 'valid_decimals']]

# Print the filtered rows in a readable format
for idx, row in corpus_filtered.iterrows():
    print(f"Row index {idx} contains decimals: {row['valid_decimals']}")
    print(f"Line: {row['Data']}\n")
    print("-------------------------------------------------------")


Row index 2388 contains decimals: ['99.1']
Line: Don Hladiuk is Calgary's eye on astronomy & space science news. On the CBC EyeOpener 1010AM or 99.1FM on the 1st or 2nd Monday of the month at 7:36 AM MT.

-------------------------------------------------------
Row index 2718 contains decimals: ['99.1']
Line: Don Hladiuk is Calgary's eye on astronomy & space science news. On the CBC EyeOpener 1010AM or 99.1FM on the 1st or 2nd Monday of the month at 7:36 AM MT.

-------------------------------------------------------
Row index 3208 contains decimals: ['106.1']
Line: AIR 106.1 FM + DiscoverAirdrie Local News.
Want to win a truck? Our Ram Everyday Adventure Contest is BACK!

@RadioClaireMC

-------------------------------------------------------
Row index 3663 contains decimals: ['26.4']
Line: hier schreiben die Hühner aus dem Stall  (*25./26.4. 2019) . 
Körner und Würmer immer willkommen

-------------------------------------------------------
Row index 3861 contains decimals: ['106.1']


In [54]:
print(f"Total rows with standalone decimals: {len(corpus_filtered)}")


Total rows with standalone decimals: 22


## prompts
First: how would you defind a decimal number, would you consider integers or stictly those with a decimal point?

Last: suggest me a way to properly display the list of such rows.

## Q.6  What is the total number of ip addresses across all the records?

In [55]:
# Regular expression to match IPv4 addresses (e.g., 192.168.1.1)
ip_pattern = r'''
(?:
  (?:25[0-5]        # 250–255
  | 2[0-4][0-9]     # 200–249
  | 1[0-9]{2}       # 100–199
  | [1-9]?[0-9])    # 0–99
  \.
){3}
(?:25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])
'''



# Compile with VERBOSE to support comments and formatting
compiled_ip_pattern = re.compile(ip_pattern, re.VERBOSE)

# Function to extract strictly valid IPv4 addresses
def extract_strict_ips(text):
    return compiled_ip_pattern.findall(text)

# Apply to DataFrame
corpus['ip_addresses'] = corpus['Data'].apply(extract_strict_ips)
corpus['ip_count'] = corpus['ip_addresses'].apply(len)



In [51]:
#total IP count
total_ips = df['ip_count'].sum()
print(f"Total valid IPv4 addresses: {total_ips}\n")

#Show rows that contain IPs
df_with_ips = df[df['ip_count'] > 0][['Data', 'ip_addresses']]

for idx, row in df_with_ips.iterrows():
    print(f"Row index {idx} contains IPs: {row['ip_addresses']}")
    print(f"Line: {row['Data']}\n")

#Get a flat list of all IPs across all rows
all_ips = [ip for sublist in df['ip_addresses'] for ip in sublist]
print("List of all valid IP addresses:")
print(all_ips)

Total valid IPv4 addresses: 0

List of all valid IP addresses:
[]


## Prompts
First: what is the proper format of ipv4 addresses?

Last: Multiple lines in regular expression